# Construct a statistically large test dataset of top 50 frequent diseases in our dataset to compare geometry (topology and manifold properties)

In [24]:
# Load Meta Data
import os, json
from tqdm import tqdm
import numpy as np, pandas as pd
project_path = '/media/lleger/LaCie/mit/disease_geometry/'
top_diseases = json.load(open(project_path + 'diseases.json'))['diseases']; print(len(top_diseases), "diseases")

import sys
sys.path.append('../../')
from polygene.data_utils.tokenization import normalise_str
age_map = json.load(open('../data_utils/vocab/age_map.json'))
is_adult = lambda development_stage: age_map.get(normalise_str(development_stage)[1:-1], 40) >= 30

METADATA_PATH = '/media/rohola/ssd_storage/primary_metadata/'
metadata = pd.concat([pd.read_pickle(METADATA_PATH+f) for f in sorted(os.listdir(METADATA_PATH))], ignore_index=True)
metadata['adult'] = metadata['development_stage'].apply(is_adult)
normal_cell_types = metadata[metadata['disease'] == "normal"].value_counts(['tissue_general', 'cell_type']).reset_index()
metadata_non_immune_cells = metadata[~metadata["cell_type"].str.contains('|'.join(['T cell', 'B cell', 'killer', 'monocyte', 'macrophage', 'plasma cell']), case=False)]

50 diseases


/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
disease_cell_type = {}
disease_tissue = set()
for disease in tqdm(top_diseases):
    if disease == "normal": continue
    tissue = metadata_non_immune_cells[metadata_non_immune_cells['disease'] == disease].value_counts(['tissue_general', 'cell_type']).index.tolist()[0][0]
    disease_metadata = metadata[(metadata['disease'] == disease) & (metadata['tissue_general'] == tissue)]

    adult_status = 
    cell_types = disease_metadata.value_counts('cell_type').reset_index()
    
    immune_cell_mask = cell_types["cell_type"].str.contains('|'.join(['T cell', 'B cell', 'killer', 'monocyte', 'macrophage', 'plasma cell']), case=False)
    malignant_cell = cell_types['cell_type'] == "malignant cell" #malignant endswith T cell
    immune_cell_mask = ~malignant_cell & immune_cell_mask

    disease_cell_type[disease] = []
    disease_cell_type[disease].append((cell_types[~immune_cell_mask]['cell_type'].tolist()[0], tissue))

    if cell_types[immune_cell_mask]['cell_type'].tolist():
        disease_cell_type[disease].append((cell_types[immune_cell_mask]['cell_type'].tolist()[0], tissue)) 
    disease_tissue.add(tissue)

disease_cell_type['normal'] = []
print("taking normal cells for non duplicate tissues", disease_tissue)
for tissue in disease_tissue:
    normal_cell_type_of_tissue = normal_cell_types[normal_cell_types['tissue_general'] == tissue]
    immune_cells = normal_cell_type_of_tissue['cell_type'].str.contains('|'.join(['T cell', 'B cell', 'killer', 'monocyte', 'macrophage', 'plasma cell']), case=False)
    # take an immune and tissue specific cell for each tissue type
    disease_cell_type['normal'].append((normal_cell_type_of_tissue[~immune_cells]['cell_type'].tolist()[0], tissue))
    disease_cell_type['normal'].append((normal_cell_type_of_tissue[immune_cells]['cell_type'].tolist()[0], tissue))

json.dump(disease_cell_type, open(project_path + "disease_cell_type.json", 'w'))

100%|██████████| 50/50 [02:38<00:00,  3.18s/it]

taking normal cells for non duplicate tissues {'liver', 'bone marrow', 'nose', 'kidney', 'heart', 'lymph node', 'lung', 'blood', 'breast', 'stomach', 'small intestine', 'brain', 'mucosa', 'prostate gland'}


In [ ]:
import json # results of metadata identification
project_path = '/media/lleger/LaCie/mit/disease_geometry/'
json.load(open(project_path + "disease_cell_type.json"))

[['enterocyte of colon', 'small intestine'],
 ['colon goblet cell', 'small intestine']]

In [ ]:
import os, sys
sys.path.append("../../")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, scanpy as sc
import json
from tqdm import tqdm
from anndata import AnnData
from polygene.data_utils.tokenization import normalise_str

DATASET_PATH = '/media/rohola/ssd_storage/primary/'
SAVE_PATH = '/media/lleger/LaCie/mit/disease_geometry/dataset/'
os.makedirs(SAVE_PATH, exist_ok=True)
test_shard_idx = (2503, 3503)
dataset_files = [DATASET_PATH + f'cxg_chunk{i}.h5ad' for i in range(*test_shard_idx)]
number_of_cells = int(5e3)

top_diseases = json.load(open('/media/lleger/LaCie/mit/disease_geometry/diseases.json'))['diseases']
cell_types_per_disease = json.load(open("/media/lleger/LaCie/mit/disease_geometry/disease_cell_type.json"))

age_map = json.load(open('../data_utils/vocab/age_map.json'))
is_adult = lambda development_stage: age_map.get(normalise_str(development_stage)[1:-1], 40) >= 30

for disease in top_diseases:
          
    cell_types = cell_types_per_disease[disease]
    required_cells = {(disease, cell_type, tissue, True): number_of_cells for cell_type, tissue in cell_types}

    total_cells = 0
    anndata_slices = []
    progressbar = tqdm(dataset_files, desc=disease)
    included_phenotypes = ['disease', 'cell_type', 'tissue_general', 'adult']

    for file_path in progressbar:
        progressbar.set_description(f"{disease}: {total_cells} cells")
        cxg_shard = sc.read_h5ad(file_path)
        cxg_shard.obs['adult'] = cxg_shard.obs['development_stage'].apply(is_adult)
        for phenotype_combination in list(required_cells.keys()):
            mask = np.logical_and.reduce([cxg_shard.obs[phenotype] == value for phenotype, value in zip(included_phenotypes, phenotype_combination)])
            if not mask.any(): continue

            filtered_chunk = cxg_shard[mask][:required_cells[phenotype_combination]].copy()
            anndata_slice = AnnData(X=filtered_chunk.X.copy(), obs=filtered_chunk.obs.copy(), var=filtered_chunk.var.copy())
            
            anndata_slices.append(anndata_slice)
            total_cells += anndata_slice.n_obs
            required_cells[phenotype_combination] -= anndata_slice.n_obs
            
            if required_cells[phenotype_combination] <= 0: del required_cells[phenotype_combination]
            del filtered_chunk
        del cxg_shard
        if not required_cells: break

    shard = sc.concat(anndata_slices)
    shard.obs.index.name = None
    shard.write(f"{SAVE_PATH + disease}_shard.h5ad")
    del shard

normal: 0 cells:   0%|          | 0/1000 [00:00<?, ?it/s]

             adult                         development_stage  \
soma_joinid                                                    
54611220      True                   37-year-old human stage   
58323557     False  10th week post-fertilization human stage   
47558907      True                   67-year-old human stage   
46566331     False                    1-year-old human stage   
54505528      True                   51-year-old human stage   
...            ...                                       ...   
39302787     False  17th week post-fertilization human stage   
25542470      True                   50-year-old human stage   
26202659      True                   50-year-old human stage   
3019741      False                   3-month-old human stage   
49221447      True                   49-year-old human stage   

                                                   norm  
soma_joinid                                              
54611220                      [37_year_old_human_st

ValueError: 

In [ ]:
import os, sys
sys.path.append("../../")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, scanpy as sc
import json
from tqdm import tqdm
from anndata import AnnData
from polygene.data_utils.tokenization import normalise_str

DATASET_PATH = '/media/rohola/ssd_storage/primary/'
SAVE_PATH = '/media/lleger/LaCie/mit/disease_geometry/dataset/'
os.makedirs(SAVE_PATH, exist_ok=True)
TEST_CHUNK_IDS = (2503, 3570)
dataset_files = [DATASET_PATH + f'cxg_chunk{i}.h5ad' for i in range(*TEST_CHUNK_IDS)]
number_of_cells = int(5e3)

disease_tissue = json.load(open("/media/lleger/LaCie/mit/disease_geometry/disease_tissues.json"))
top_diseases = json.load(open('/media/lleger/LaCie/mit/disease_geometry/diseases.json'))['diseases']
cell_types_per_disease = json.load(open("/media/lleger/LaCie/mit/disease_geometry/cell_types_per_disease.json"))

age_map = json.load(open('../data_utils/vocab/age_map.json'))
is_adult = lambda row: "yes" if age_map.get(normalise_str(row['development_stage']), 40) // 10 >= 4 else "no"

for disease in top_diseases:
    if disease == "normal": continue
    
    tissue = disease_tissue[disease]
    cell_types = cell_types_per_disease[disease]
    required_cells = {(disease, cell_type, tissue, 'yes'): number_of_cells for cell_type in cell_types}.update(
        {("normal", cell_type, tissue, 'yes'): number_of_cells for cell_type in cell_types})
    
    total_cells = 0
    anndata_slices = []
    progressbar = tqdm(np.random.permutation(dataset_files), desc=disease)
    included_phenotypes = ['disease', 'cell_type', 'tissue_general', 'adult']

    for file_path in progressbar:
        progressbar.set_description(f"{disease}: {total_cells} cells")
        loaded_chunk = sc.read_h5ad(file_path)
        loaded_chunk.obs['adult'] = loaded_chunk.obs.apply(is_adult, axis=1)

        for phenotype_combination in list(required_cells.keys()):
            mask = np.logical_and.reduce([ loaded_chunk.obs[phenotype] == value 
                for phenotype, value in zip(included_phenotypes, phenotype_combination)])

            if not mask.any(): continue
            filtered_chunk = loaded_chunk[mask][:required_cells[phenotype_combination]].copy()
            anndata_slice = AnnData(X=filtered_chunk.X.copy(), obs=filtered_chunk.obs.copy(), var=filtered_chunk.var.copy())
            anndata_slices.append(anndata_slice)
            total_cells += anndata_slice.n_obs
            required_cells[phenotype_combination] -= anndata_slice.n_obs

            if required_cells[phenotype_combination] <= 0: del required_cells[phenotype_combination]
            del filtered_chunk
        del loaded_chunk
        if not required_cells: break

    shard = sc.concat(anndata_slices)
    shard.write(f"{SAVE_PATH + disease}_shard.h5ad")
    del shard

## isolate cells for our specific case study of lung cancer

In [11]:
import os, json
import numpy as np, pandas as pd
project_path = '/media/lleger/LaCie/mit/disease_geometry/'
top_diseases = json.load(open(project_path + 'diseases.json'))['diseases']
lung_cancers = [disease for disease in top_diseases if "lung" in disease and "carcinoma" in disease and not "non" in disease]
print(lung_cancers)
METADATA_PATH = '/media/rohola/ssd_storage/primary_metadata/'
metadata = pd.concat([pd.read_pickle(METADATA_PATH+f) for f in sorted(os.listdir(METADATA_PATH))], ignore_index=True)

['lung adenocarcinoma', 'squamous cell lung carcinoma', 'small cell lung carcinoma', 'lung large cell carcinoma']


In [ ]:
# hand pick cell types for comparison
cell_type_selection = {
            "small cell lung carcinoma": ['epithelial cell', 'T cell'],
            "lung large cell carcinoma": ['native cell', 'CD8-positive, alpha-beta T cell', 'CD4-positive, alpha-beta T cell'],
            "lung adenocarcinoma": ['CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell', 'alveolar macrophage', 'malignant cell',],
            "squamous cell lung carcinoma": ['malignant cell', 'CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell', 'alveolar macrophage'], 
            'normal': ['alveolar macrophage', 'epithelial cell of alveolus of lung', 'type II pneumocyte', 'CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell']
            }

In [ ]:
import os, sys
sys.path.append("../../")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, scanpy as sc
import json
from tqdm import tqdm
from anndata import AnnData
from polygene.data_utils.tokenization import normalise_str

DATASET_PATH = '/media/rohola/ssd_storage/primary/'
SAVE_PATH = '/media/lleger/LaCie/mit/disease_geometry/lung_dataset/'
os.makedirs(SAVE_PATH, exist_ok=True)
TEST_CHUNK_IDS = (2503, 3570)
dataset_files = [DATASET_PATH + f'cxg_chunk{i}.h5ad' for i in range(*TEST_CHUNK_IDS)]
number_of_cells = int(5e3)

disease_tissue = json.load(open("/media/lleger/LaCie/mit/disease_geometry/disease_tissues.json"))
top_diseases = json.load(open('/media/lleger/LaCie/mit/disease_geometry/diseases.json'))['diseases']

age_map = json.load(open('../data_utils/vocab/age_map.json'))
is_adult = lambda row: "yes" if age_map.get(normalise_str(row['development_stage']), 40) // 10 >= 4 else "no"

for disease in cell_type_selection:
    if disease == "normal": continue
    
    tissue = disease_tissue[disease]
    cell_types = cell_type_selection[disease]
    required_cells = {(disease, cell_type, tissue, 'yes'): number_of_cells for cell_type in cell_types}.update(
        {("normal", cell_type, tissue, 'yes'): number_of_cells for cell_type in cell_types})
    
    total_cells = 0
    anndata_slices = []
    progressbar = tqdm(np.random.permutation(dataset_files), desc=disease)
    included_phenotypes = ['disease', 'cell_type', 'tissue_general', 'adult']

    for file_path in progressbar:
        progressbar.set_description(f"{disease}: {total_cells} cells")
        loaded_chunk = sc.read_h5ad(file_path)
        loaded_chunk.obs['adult'] = loaded_chunk.obs.apply(is_adult, axis=1)

        for phenotype_combination in list(required_cells.keys()):
            mask = np.logical_and.reduce([ loaded_chunk.obs[phenotype] == value 
                for phenotype, value in zip(included_phenotypes, phenotype_combination)])

            if not mask.any(): continue
            filtered_chunk = loaded_chunk[mask][:required_cells[phenotype_combination]].copy()
            anndata_slice = AnnData(X=filtered_chunk.X.copy(), obs=filtered_chunk.obs.copy(), var=filtered_chunk.var.copy())
            anndata_slices.append(anndata_slice)
            total_cells += anndata_slice.n_obs
            required_cells[phenotype_combination] -= anndata_slice.n_obs

            if required_cells[phenotype_combination] <= 0: del required_cells[phenotype_combination]
            del filtered_chunk
        del loaded_chunk
        if not required_cells: break

    shard = sc.concat(anndata_slices)
    shard.write(f"{SAVE_PATH + disease}_shard.h5ad")
    del shard